In [0]:
%sql
---- Creating new catalog, schema -----
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


##### Pareto Analysis (80/20 Rule)

Concept:
- The Pareto Principle (80/20 Rule) states that approximately 80% of the outcome comes from 20% of the causes.
In business, this often means a small number of products generate most of the revenue.

Question:
- Find the products contributing to the first 80% of total sales.

Idea:
- Aggregate sales by product.
- Calculate cumulative sales.
- Keep products until cumulative sales reaches 80% of total sales.

Concepts:
GROUP BY | Window Function | Running Total | CTE

In [0]:
%sql
with product_sales as (
    select
        Product_ID,
        sum(sales) as total_sales
    from superstore_orders
    group by Product_ID
),
running_total as (
    select
        Product_ID,
        sum(total_sales) over (order by total_sales desc, Product_ID rows between unbounded preceding and current row) as cumulative_sales,
        0.8*sum(total_sales) over () as ultimate_sales
    from product_sales
)
select
    Product_ID,
    cumulative_sales,
    ultimate_sales
from running_total
where cumulative_sales <= ultimate_sales;

Product_ID,cumulative_sales,ultimate_sales
TEC-CO-10004722,61599.824,1837760.6882399973
OFF-BI-10003527,89053.208,1837760.6882399973
TEC-MA-10002412,111691.688,1837760.6882399973
FUR-CH-10002024,133562.264,1837760.6882399973
OFF-BI-10001359,153385.743,1837760.6882399973
OFF-BI-10000545,172410.243,1837760.6882399973
TEC-CO-10001449,191249.929,1837760.6882399973
TEC-MA-10001127,209624.824,1837760.6882399973
OFF-BI-10004995,227589.892,1837760.6882399973
OFF-SU-10000151,244620.204,1837760.6882399973


##### 01. Analyze customers' orders and products purchased to identify relationships or purchasing patterns.

In [0]:
%sql
DROP TABLE IF EXISTS customer_orders;
DROP TABLE IF EXISTS customer_products;
CREATE TABLE customer_orders (order_id INT, customer_id INT, product_id INT);
INSERT INTO customer_orders VALUES
(1, 1, 1),
(1, 1, 2),
(1, 1, 3),
(2, 2, 1),
(2, 2, 2),
(2, 2, 4),
(3, 1, 5);
CREATE TABLE customer_products (id INT, name STRING);
INSERT INTO customer_products VALUES
(1, 'A'),
(2, 'B'),
(3, 'C'),
(4, 'D'),
(5, 'E');
SELECT * FROM customer_orders;

order_id,customer_id,product_id
1,1,1
1,1,2
1,1,3
2,2,1
2,2,2
2,2,4
3,1,5


In [0]:
%sql
SELECT * FROM customer_products;

id,name
1,A
2,B
3,C
4,D
5,E


In [0]:
%sql
with cte as (
select 
    o1.product_id as a, 
    o2.product_id as b,
    count(*) as freq
from customer_orders as o1
    inner join
customer_orders as o2
on o1.order_id = o2.order_id
where o1.product_id < o2.product_id
group by o1.product_id, o2.product_id
)

select 
    concat_ws(' ', c1.name, c2.name) as products,
    freq
from cte
join customer_products as c1
on cte.a = c1.id
join customer_products as c2
on cte.b = c2.id;

products,freq
A C,1
A D,1
B C,1
A B,2
B D,1


##### 02. Given the 2 tables, return the fraction of users, rounded to 2 decimal places, who accessed Amazon Music and upgraded to Prime Membership within the first 30 days of signing up.

In [0]:
%sql
DROP TABLE IF EXISTS app_users;
DROP TABLE IF EXISTS app_events;

CREATE TABLE app_users (
    user_id INT,
    name STRING,
    join_date DATE
);

INSERT INTO app_users VALUES
(1, 'Jon', DATE('2020-02-14')),
(2, 'Jane', DATE('2020-02-14')),
(3, 'Jill', DATE('2020-02-15')),
(4, 'Josh', DATE('2020-02-15')),
(5, 'Jean', DATE('2020-02-16')),
(6, 'Justin', DATE('2020-02-17')),
(7, 'Jeremy', DATE('2020-02-18'));

CREATE TABLE app_events (
    user_id INT,
    type STRING,
    access_date DATE
);

INSERT INTO app_events VALUES
(1, 'Pay',   DATE('2020-03-01')),
(2, 'Music', DATE('2020-03-02')),
(2, 'P',     DATE('2020-03-12')),
(3, 'Music', DATE('2020-03-15')),
(4, 'Music', DATE('2020-03-15')),
(1, 'P',     DATE('2020-03-16')),
(3, 'P',     DATE('2020-03-22'));

SELECT * FROM app_users;

user_id,name,join_date
1,Jon,2020-02-14
2,Jane,2020-02-14
3,Jill,2020-02-15
4,Josh,2020-02-15
5,Jean,2020-02-16
6,Justin,2020-02-17
7,Jeremy,2020-02-18


In [0]:
%sql
SELECT * FROM app_events;

user_id,type,access_date
1,Pay,2020-03-01
2,Music,2020-03-02
2,P,2020-03-12
3,Music,2020-03-15
4,Music,2020-03-15
1,P,2020-03-16
3,P,2020-03-22


In [0]:
%sql
with cte as (
select
    u.*,
    e.type,
    e.access_date,
    date_diff(day, u.join_date, e.access_date) as no_of_days
from app_users as u
left join app_events as e
on u.user_id = e.user_id and e.type = "P"
where u.user_id in (
    select user_id from app_events where type = "Music")
)
select
    count(distinct(user_id)) as total_no_of_users,
    sum(
        case
            when no_of_days <= 30 then 1
            else 0
        end
    ) as users_within_30_days,
    round((sum(case when no_of_days <= 30 then 1 end))/(count(distinct(user_id)))*100,2) as conversion_rate
from cte;

total_no_of_users,users_within_30_days,conversion_rate
3,1,33.33


#### 03. Calculate customer retention and churn metrics based on customers' purchase history.
###### Identify recurring (retained) customers and churned customers based on their previous and next orders.

In [0]:
%sql
DROP TABLE IF EXISTS customer_transactions;
CREATE TABLE customer_transactions (
    order_id INT,
    cust_id INT,
    order_date DATE,
    amount INT
);

INSERT INTO customer_transactions VALUES
(1, 1, '2020-01-15', 150),
(2, 1, '2020-02-10', 150),
(3, 2, '2020-01-16', 150),
(4, 2, '2020-02-25', 150),
(5, 3, '2020-01-10', 150),
(6, 3, '2020-02-20', 150),
(7, 4, '2020-01-20', 150),
(8, 5, '2020-02-20', 150);

SELECT * FROM customer_transactions ORDER BY order_id;

order_id,cust_id,order_date,amount
1,1,2020-01-15,150
2,1,2020-02-10,150
3,2,2020-01-16,150
4,2,2020-02-25,150
5,3,2020-01-10,150
6,3,2020-02-20,150
7,4,2020-01-20,150
8,5,2020-02-20,150


In [0]:
%sql
with cte as (
    select
        *,
        lag(order_date) over(partition by cust_id order by order_date) as prev_order_date,
        lead(order_date) over(partition by cust_id order by order_date) as next_order_date
    from customer_transactions
)
select
    extract(month from order_date) as order_month,
    sum(case
            when (extract(month from order_date))-(extract(month from prev_order_date)) = 1 then 1
            else 0
        end) as recurring,
    sum(case
            when (prev_order_date is null) and (next_order_date is null) then 1
            else 0
    end) as churned,
    count(*) as no_of_customers
from cte
group by extract(month from order_date);


order_month,recurring,churned,no_of_customers
1,0,1,4
2,3,1,4


#### 04. Find the second most recent activity for each user; if a user has only one activity, return that activity.

In [0]:
%sql
DROP TABLE IF EXISTS customer_activity;
CREATE TABLE customer_activity (
    username VARCHAR(20),
    activity VARCHAR(20),
    startDate DATE,
    endDate DATE
);
INSERT INTO customer_activity VALUES
('Alice', 'Travel',  '2020-02-12', '2020-02-20'),
('Alice', 'Dancing', '2020-02-21', '2020-02-23'),
('Alice', 'Travel',  '2020-02-24', '2020-02-28'),
('Bob',   'Travel',  '2020-02-11', '2020-02-18');
SELECT * FROM customer_activity;

username,activity,startDate,endDate
Alice,Travel,2020-02-12,2020-02-20
Alice,Dancing,2020-02-21,2020-02-23
Alice,Travel,2020-02-24,2020-02-28
Bob,Travel,2020-02-11,2020-02-18


In [0]:
%sql
-------------------------------------------  Workaround 1  ---------------------------------------------
with cte as(
    select
        *,
        rank() over(partition by username order by endDate desc) as rank,
        count(*) over(partition by username) as total_users
    from customer_activity
)
select
    *
from cte where rank = 2 or total_users = 1;


-------------------------------------------- Workaround 2 ----------------------------------------------
-- with cte as (
--     select *
--     from customer_activity where username in (
--         select username from customer_activity group by username having count(*) = 1
--     )
-- ),
-- cte1 as (
--         select 
--             *, 
--             rank() over(partition by username order by endDate desc) as rank
--         from customer_activity
-- )
-- select * from cte
-- union
-- select username, activity, startDate, endDate from cte1 where rank = 2;

username,activity,startDate,endDate,rank,total_users
Alice,Dancing,2020-02-21,2020-02-23,2,3
Bob,Travel,2020-02-11,2020-02-18,1,1


username,activity,startDate,endDate
Bob,Travel,2020-02-11,2020-02-18
Alice,Dancing,2020-02-21,2020-02-23


##### 05. Calculate each employee's total bill by applying the billing rate that was effective on each employee's work date.

In [0]:
%sql
DROP TABLE IF EXISTS employee_billings;
DROP TABLE IF EXISTS employee_hoursworked;
CREATE TABLE employee_billings (
    emp_name VARCHAR(10),
    bill_date DATE,
    bill_rate INT
);
INSERT INTO employee_billings VALUES
('Sachin', '1990-01-01', 25),
('Sehwag', '1989-01-01', 15),
('Dhoni', '1989-01-01', 20),
('Sachin', '1991-02-05', 30);


CREATE TABLE employee_hoursworked (
    emp_name VARCHAR(20),
    work_date DATE,
    bill_hrs INT
);
INSERT INTO employee_hoursworked VALUES
('Sachin', '1990-07-01', 3),
('Sachin', '1990-08-01', 5),
('Sehwag', '1990-07-01', 2),
('Sachin', '1991-07-01', 4);

SELECT * FROM employee_billings;

emp_name,bill_date,bill_rate
Sachin,1990-01-01,25
Sehwag,1989-01-01,15
Dhoni,1989-01-01,20
Sachin,1991-02-05,30


In [0]:
%sql
SELECT * FROM employee_hoursworked;

emp_name,work_date,bill_hrs
Sachin,1990-07-01,3
Sachin,1990-08-01,5
Sehwag,1990-07-01,2
Sachin,1991-07-01,4


In [0]:
%sql
with cte as (
    select
        *,
        lead(to_date(dateadd(day, -1, bill_date)), 1, '9999-12-31') over(partition by emp_name order by bill_date) as next_bill_date
    from employee_billings
),
cte1 as(
    select 
        hw.*, 
        cte.bill_rate, 
        cte.bill_date, 
        cte.next_bill_date
    from cte
    inner join employee_hoursworked as hw 
    on cte.emp_name = hw.emp_name 
    where hw.work_date between cte.bill_date and cte.next_bill_date
)
select 
    cte1.emp_name,
    sum(cte1.bill_hrs * cte1.bill_rate) as total_bill
from cte1
group by cte1.emp_name;

emp_name,total_bill
Sachin,320
Sehwag,30
